# Retired Ensembl gene identifiers in an HCA integrated atlas

## The problem

An integrated atlas is built from studies that were each aligned against whatever Ensembl
annotation release was current at the time. Those releases disagree. Over the years Ensembl
deletes some gene models, renames others, and merges neighbouring ones together.

When an atlas is assembled as the **union** of its source datasets, it inherits every one of
those disagreements. The result is a gene axis containing identifiers that no longer exist.

This matters in two quite different ways.

**For submission.** CZ CELLxGENE requires every identifier in a dataset to resolve against
current GENCODE. A retired identifier is a validation **error**, not a warning, and blocks
submission until it is resolved.

**For the atlas itself.** Where Ensembl has merged two genes that a study counted separately,
the atlas ends up holding *two columns for what is now one gene*. Any per-column statistic —
including a field like `n_studies_detected` — then describes each half rather than the whole.

## What this notebook does

It takes every retired identifier in an atlas and works out which of three things happened to
it, because each needs a different action:

| | What happened | What it needs |
|---|---|---|
| **A** | Deleted — no replacement exists | Remove the column |
| **B** | Renamed — replacement is not in the atlas | Rename the column |
| **C** | Merged — replacement is *already* a column | A decision |

It then checks that classification rather than trusting it: confirming the merges against
genomic coordinates, and working out which source study contributed each identifier.

Everything is **read-only**. No `.h5ad` file is modified.

## Requirements

```
pip install anndata pandas pymysql scipy jupyter
```

`pymysql` queries Ensembl's public MySQL server at `ensembldb.ensembl.org`, which needs
outbound access on port 3306. Results are cached to disk on first use, so a second run needs
no network. The atlas is opened in `anndata`'s backed mode, so the expression matrix is never
loaded into memory.

## 1. Configuration

Set the paths for the atlas you want to analyse, then run the notebook top to bottom.

`OLD_RELEASE` should be an Ensembl release the source studies plausibly used — it is the
"before" picture used to confirm merges in section 5. If you do not know it, section 8 will
tell you, and you can come back and adjust it.

In [ ]:
from pathlib import Path

# ---- EDIT THESE -----------------------------------------------------------
ATLAS = Path("/path/to/integrated-objects/my-atlas.h5ad")

# Source datasets the atlas was built from. Set to None to skip sections 6-8.
SOURCES_GLOB = "/path/to/source-datasets/*.h5ad"

# Example — breast v1 (uncomment to reproduce):
# BREAST = Path("~/hca-tracker-upload/prod/breast/breast-v1").expanduser()
# ATLAS        = BREAST / "integrated-objects/tracker-source/archive/all-breast-cells-r1-wip-7.h5ad"
# SOURCES_GLOB = str(BREAST / "source-datasets/*.h5ad")

# Current GENCODE gene list. The copy vendored in hca-schema-validator is the same
# reference CELLxGENE validates against, so it is the one to use.
GENCODE_CSV = Path(
    "/path/to/hca-schema-validator/src/hca_schema_validator/"
    "_vendored/cellxgene_schema/gencode_files/genes_homo_sapiens.csv.gz"
)

SPECIES          = "homo_sapiens"
ENSEMBL_RELEASE  = 114   # should match the release behind GENCODE_CSV
ENSEMBL_ASSEMBLY = 38
OLD_RELEASE      = 87    # a release the source studies plausibly used (see section 8)

# Releases to test in section 8. The full GRCh38 range gives an exact answer; a
# sparse sample is faster on a first run but only brackets it. Results are cached,
# so the full range costs a couple of minutes once.
RELEASES_TO_TEST = list(range(76, 117))
CACHE = Path("./ensembl_cache")
# ---------------------------------------------------------------------------

CACHE.mkdir(parents=True, exist_ok=True)
print(f"atlas        {ATLAS}")
print(f"sources      {SOURCES_GLOB}")
print(f"Ensembl      release {ENSEMBL_RELEASE}, comparing against {OLD_RELEASE}")
print(f"cache        {CACHE.resolve()}")

In [ ]:
import gzip, csv, glob, hashlib
import pandas as pd
import anndata as ad

try:
    import pymysql
except ImportError:
    raise SystemExit("pymysql is required:  pip install pymysql")

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

CORE     = f"{SPECIES}_core_{ENSEMBL_RELEASE}_{ENSEMBL_ASSEMBLY}"
OLD_CORE = f"{SPECIES}_core_{OLD_RELEASE}_{37 if OLD_RELEASE < 76 else ENSEMBL_ASSEMBLY}"

def ensembl_query(sql, params=None):
    """Query Ensembl's public MySQL server, caching on the SQL and parameters."""
    key = hashlib.sha1(f"{sql}{params}".encode()).hexdigest()[:16]
    cached = CACHE / f"q_{key}.csv"
    if cached.exists():
        return pd.read_csv(cached, dtype=str).convert_dtypes()
    conn = pymysql.connect(host="ensembldb.ensembl.org", user="anonymous",
                           password="", connect_timeout=90)
    try:
        with conn.cursor() as cur:
            cur.execute(sql, params or ())
            rows, cols = cur.fetchall(), [d[0] for d in cur.description]
    finally:
        conn.close()
    df = pd.DataFrame(rows, columns=cols)
    df.to_csv(cached, index=False)
    return df

def ensembl_query_many(sql_template, values, chunk=300):
    """Run a query over a long IN (...) list, in chunks."""
    frames = []
    values = list(values)
    for i in range(0, len(values), chunk):
        part = values[i:i + chunk]
        frames.append(ensembl_query(sql_template.format(ph=",".join(["%s"] * len(part))), part))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

print(f"ready — querying {CORE}, comparing against {OLD_CORE}")

## 2. Which identifiers are no longer current?

We read the atlas's `var` index and compare it against the current GENCODE gene list. Anything
absent from that list is an identifier CELLxGENE will reject.

The second cell asks how much data those identifiers actually carry. This is worth knowing
early: if they are all but undetected, the decisions ahead are low-stakes; if any is widely
expressed, it matters that it is renamed rather than dropped.

In [ ]:
gencode = {}
with gzip.open(GENCODE_CSV, "rt") as fh:
    for row in csv.reader(fh):
        if len(row) > 4:
            gencode[row[0]] = {"symbol": row[1], "biotype": row[4]}

adata     = ad.read_h5ad(ATLAS, backed="r")
atlas_ids = [str(i) for i in adata.var.index]

# An atlas built by unioning studies can repeat an identifier. Everything below
# indexes by identifier, so stop here rather than produce silently wrong joins.
if len(set(atlas_ids)) != len(atlas_ids):
    dupes = pd.Series(atlas_ids).value_counts()
    raise ValueError(f"var index is not unique — {int((dupes > 1).sum())} repeated identifier(s), "
                     f"e.g. {list(dupes[dupes > 1].index[:5])}")

retired = [i for i in atlas_ids if i not in gencode]

print(f"{ATLAS.name}")
print(f"  {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(f"  {len(gencode):,} genes in current GENCODE")
print()
print(f"  {len(retired):,} identifiers are no longer current "
      f"({100 * len(retired) / len(atlas_ids):.2f}% of the gene axis)")
print(f"  every one of these is a CELLxGENE validation error")

In [ ]:
if "n_cells_expressed" in adata.var.columns:
    cells_expressed = pd.Series(adata.var["n_cells_expressed"].astype(int).values, index=atlas_ids)
else:
    # Count from adata.X, whose columns are adata.var by definition. adata.raw may
    # carry a different gene set, so counting there and labelling with adata.var
    # names would attach counts to the wrong genes.
    import numpy as np, scipy.sparse as sp
    counts = np.zeros(adata.n_vars, dtype=int)
    for start in range(0, adata.n_obs, 50_000):
        b = adata.X[start:start + 50_000]
        counts += (np.asarray((b > 0).sum(axis=0)).ravel() if sp.issparse(b)
                   else (b > 0).sum(axis=0))
    cells_expressed = pd.Series(counts, index=atlas_ids)

if not retired:
    print("No retired identifiers — this atlas resolves cleanly against current GENCODE.")
    print("Sections 3 to 9 will have nothing to report.")

rc = cells_expressed.loc[retired] if retired else pd.Series(dtype=int)
print(f"Of the {len(retired):,} retired identifiers, counting cells in which each is detected:")
print()
if len(rc):
    print(f"  detected in no cells at all : {int((rc == 0).sum()):,}")
    print(f"  median                      : {int(rc.median()):,} cells")
    print(f"  most widely detected        : {int(rc.max()):,} cells "
          f"({100 * rc.max() / adata.n_obs:.1f}% of the atlas)")
print()
print("A widely detected identifier is a reason to resolve these carefully rather than")
print("simply dropping them — section 4 separates the ones that can be renamed.")

## 3. What happened to each identifier?

Ensembl's `stable_id_event` table records how identifiers map across releases. This is the
authoritative source, and better than matching on gene symbols, which change independently of
identifiers.

Two things to know about reading it:

- A row where the new identifier **equals** the old one means the identifier simply survived
  that release. Most rows are of this kind and carry no information for us.
- A row where they **differ** is a transition — the identifier was replaced by another.
- Each row carries a **score**. A real correspondence scores close to 1. A score of 0 means
  none was established, and the row is a suggestion rather than a mapping, so we ignore it.

A retired identifier with no scoring transition row was deleted outright.

In [ ]:
if not retired:
    raise SystemExit("Nothing to do — no retired identifiers.")

events = ensembl_query_many(
    f"""SELECT old_stable_id, new_stable_id, score
        FROM {CORE}.stable_id_event
        WHERE type = 'gene' AND old_stable_id IN ({{ph}})""",
    retired,
)

# Ensembl scores each mapping. A real correspondence scores close to 1; a score of 0
# means none was established, and such rows are suggestions rather than mappings.
# (Ensembl's REST archive service returns score-0 candidates too, which is why it can
# appear to find successors the database does not record.)
SCORE_FLOOR = 0.5

scored = events[(events.old_stable_id != events.new_stable_id) & events.new_stable_id.notna()]
low    = scored[scored.score.astype(float) < SCORE_FLOOR]
transitions = (scored[scored.score.astype(float) >= SCORE_FLOOR]
               [["old_stable_id", "new_stable_id"]]
               .drop_duplicates()
               .rename(columns={"old_stable_id": "retired", "new_stable_id": "successor"}))

if len(low):
    print(f"Ignored {len(low)} mapping(s) scoring below {SCORE_FLOOR} — no correspondence was")
    print("established for those, so they are treated as having no successor.\n")

no_successor = sorted(set(retired) - set(transitions.retired))

print(f"{len(events):,} event rows returned for {len(retired):,} identifiers")
print(f"  {len(events) - len(transitions):,} are the identifier surviving a release unchanged")
print(f"  {len(transitions):,} are real transitions to a different identifier")
print()
print(f"So {len(transitions):,} identifiers were replaced by something, and "
      f"{len(no_successor):,} were deleted with no replacement.")

### Did any gene split?

A split — one gene becoming two or more — would be genuinely ambiguous, because there would be
no single answer to what the old column should become. It is worth ruling out explicitly rather
than assuming.

In [ ]:
successors_each = transitions.groupby("retired")["successor"].nunique()
splits = successors_each[successors_each > 1]

if len(splits) == 0:
    print(f"No splits. All {len(successors_each):,} replaced identifiers have exactly one")
    print("successor, so there is nothing ambiguous to resolve here.")
else:
    display(transitions[transitions.retired.isin(splits.index)])
    raise ValueError(
        f"{len(splits)} identifier(s) map to more than one successor. The grouping below assumes "
        "a single successor per identifier and would double-count these. Review them by hand "
        "before continuing.")

### Did several genes merge into one?

The opposite case is common and unproblematic, but it changes the arithmetic: if three retired
identifiers all now point at the same gene, that gene is currently spread across four columns
in the atlas.

In [ ]:
incoming = transitions.groupby("successor")["retired"].nunique()
merged   = incoming[incoming > 1]

print(f"{len(incoming):,} distinct successors are named by these identifiers.")
print(f"  {int((incoming == 1).sum()):,} receive exactly one retired identifier")
print(f"  {len(merged):,} receive more than one — several old genes now being one gene")
if len(merged):
    print()
    print("Retired identifiers per successor:")
    display(incoming.value_counts().sort_index()
            .rename_axis("retired identifiers merging in").rename("successors").to_frame())
    print(f"The most collapsed case is {merged.max()} old identifiers now pointing at one gene.")

## 4. Grouping by the action needed

Before grouping, one wrinkle has to be dealt with: **a successor can itself be retired**. Ensembl
may have replaced gene A with gene B in one release and then replaced B with C in a later one.
Taking only the first hop would tell you to rename A to B — leaving an identifier CELLxGENE still
rejects.

So each identifier is followed through the chain until it reaches a gene that exists in current
GENCODE. If the chain never reaches one, the identifier belongs in group A regardless of how many
hops it took.

With that settled, the decisive question is whether the final successor is **already a column in
the atlas**. If it is not, the identifier can simply be renamed. If it is, the atlas holds two
columns where Ensembl now recognises one gene, and someone has to decide what to do.

In [ ]:
atlas_set = set(atlas_ids)
one_hop   = dict(zip(transitions.retired, transitions.successor))

def resolve(identifier):
    """Follow the chain until it reaches a gene in current GENCODE, or give up."""
    seen, current, hops = set(), identifier, 0
    while current in one_hop and current not in seen:
        seen.add(current)
        current = one_hop[current]
        hops += 1
    return (current, hops) if current in gencode else (None, hops)

resolved = pd.DataFrame(
    [(i, *resolve(i)) for i in retired], columns=["retired", "final", "hops"])
resolved["group"] = resolved.apply(
    lambda r: "A" if r.final is None else ("C" if r.final in atlas_set else "B"), axis=1)

chained = resolved[resolved.hops > 1]
if len(chained):
    print(f"{len(chained)} identifier(s) needed more than one hop to reach a current gene. "
          f"Stopping at the first hop would have left them unresolved:")
    display(chained)
else:
    print("No multi-hop chains: every identifier reaches a current gene in one step.")

group_a = sorted(resolved.loc[resolved.group == "A", "retired"])
group_b = sorted(resolved.loc[resolved.group == "B", "retired"])
group_c = sorted(resolved.loc[resolved.group == "C", "retired"])

def stats(ids):
    s = cells_expressed.loc[ids] if ids else pd.Series(dtype=int)
    return {"identifiers": len(ids),
            "median cells": int(s.median()) if len(s) else 0,
            "most detected": int(s.max()) if len(s) else 0}

summary = pd.DataFrame([
    {"group": "A — deleted, no replacement",      **stats(group_a), "action": "remove the column"},
    {"group": "B — renamed, successor absent",    **stats(group_b), "action": "rename the column"},
    {"group": "C — merged, successor present",    **stats(group_c), "action": "needs a decision"},
])
display(summary)

print(f"{len(group_a):,} to remove, {len(group_b):,} to rename, and {len(group_c):,} needing a decision.")
if group_b:
    top_b = cells_expressed.loc[group_b].idxmax()
    print(f"\nThe most widely detected renameable identifier is {top_b} "
          f"({cells_expressed[top_b]:,} cells), which becomes "
          f"{transitions.set_index('retired').loc[top_b, 'successor']} "
          f"({gencode.get(transitions.set_index('retired').loc[top_b, 'successor'], {}).get('symbol')}). "
          f"Dropping rather than renaming it would lose real data.")

In [ ]:
classification = resolved.rename(columns={"final": "successor"}).copy()
classification["successor_symbol"] = classification.successor.map(
    lambda s: gencode.get(s, {}).get("symbol") if s else None)
classification["cells_expressed"] = classification.retired.map(cells_expressed)
classification = classification.sort_values("cells_expressed", ascending=False)

# Deliberately not written to disk yet — sections 5 and 6 add evidence columns,
# and a curator acting on this file needs those. The CSV is written in section 9.
display(classification.head(12))

## 5. Are the group C cases really merges?

The classification above says these identifiers were replaced by a gene already in the atlas.
That is a claim about Ensembl's bookkeeping. It is worth checking against the genome itself.

If two identifiers really describe the same piece of DNA, the retired gene's position in an
older release should fall **inside** the span of the gene that replaced it. We compare
coordinates in `OLD_RELEASE` against the current release.

A few genes are revised — extended, trimmed, or moved — before they are merged, so comparing
only the first and last release can make a genuine merge look wrong. The next cell follows the
release history of anything that does not look contained, rather than leaving it unexplained.

In [ ]:
pairs_c = (resolved.loc[resolved.group == "C", ["retired", "final"]]
           .rename(columns={"final": "successor"}).reset_index(drop=True))

def coordinates(ids, core):
    # Restrict to the primary assembly. A gene also placed on an ALT or patch
    # scaffold would otherwise contribute extra rows, and keeping an arbitrary one
    # produces spurious "different chromosome" verdicts.
    df = ensembl_query_many(
        f"""SELECT g.stable_id, s.name AS chrom, g.seq_region_start AS start,
                   g.seq_region_end AS end
            FROM {core}.gene g
            JOIN {core}.seq_region s  ON g.seq_region_id = s.seq_region_id
            JOIN {core}.coord_system c ON s.coord_system_id = c.coord_system_id
            WHERE c.name = 'chromosome'
              AND c.attrib LIKE '%%default_version%%'
              AND g.stable_id IN ({{ph}})""",
        ids,
    )
    return df.drop_duplicates(subset="stable_id").set_index("stable_id")

old_pos, new_pos = coordinates(pairs_c.retired, OLD_CORE), coordinates(pairs_c.successor, CORE)

def verdict(row):
    if row.retired not in old_pos.index or row.successor not in new_pos.index:
        return "not in both releases"
    a, b = old_pos.loc[row.retired], new_pos.loc[row.successor]
    if a.chrom != b.chrom:                       return "different chromosome"
    if b.start <= a.start and a.end <= b.end:    return "contained"
    if not (a.end < b.start or b.end < a.start): return "partial overlap"
    return "no overlap"

coord_check = pairs_c.assign(verdict=pairs_c.apply(verdict, axis=1))
counts = coord_check.verdict.value_counts()
display(counts.rename("pairs").to_frame())

resolvable = int(counts.drop("not in both releases", errors="ignore").sum())
contained  = int(counts.get("contained", 0))
print(f"Of the {len(pairs_c):,} group C pairs, {resolvable:,} could be located in both releases.")
print(f"{contained:,} of those sit entirely inside the gene that replaced them, which is what a")
print("merge looks like on the genome.")

In [ ]:
odd = coord_check[~coord_check.verdict.isin(["contained", "not in both releases"])]
if odd.empty:
    print("Every locatable pair was contained — nothing further to explain.")
else:
    print(f"{len(odd)} pair(s) did not look contained. Following their release history:\n")
    for r in odd.itertuples():
        hist = ensembl_query(
            f"""SELECT e.old_stable_id, e.new_stable_id, e.score,
                       m.old_release, m.new_release
                FROM {CORE}.stable_id_event e
                JOIN {CORE}.mapping_session m
                  ON e.mapping_session_id = m.mapping_session_id
                WHERE e.type = 'gene' AND e.old_stable_id = %s
                ORDER BY m.created""", (r.retired,))
        hist = hist[hist.old_stable_id != hist.new_stable_id]
        print(f"{r.retired} -> {r.successor}  ({r.verdict})")
        display(hist)
        print("A gene that was revised or relocated before merging will not appear contained")
        print("when only the first and last release are compared.\n")

## 6. Which studies contributed each identifier?

This decides how much thought group C really needs.

If a **single source dataset contains both** members of a pair, then at the time that study was
aligned the two were separate genes in its annotation, and the aligner counted them separately.
Its cells can legitimately have counts in both columns — different reads, different loci.

If **no dataset contains both**, the two columns cannot overlap at all, and combining them is
simple addition with nothing to consider.

In [ ]:
if SOURCES_GLOB is None:
    sources = {}
    print("SOURCES_GLOB is not set — skipping sections 6 to 8.")
else:
    sources = {}
    for path in sorted(glob.glob(SOURCES_GLOB)):
        sa = ad.read_h5ad(path, backed="r")
        declared = (",".join(sorted(set(sa.obs["gene_annotation_version"].astype(str))))
                    if "gene_annotation_version" in sa.obs.columns else None)
        sources[Path(path).stem] = {"genes": {str(i) for i in sa.var.index},
                                    "n_obs": sa.n_obs, "declared": declared}
        sa.file.close()
    display(pd.DataFrame([{"dataset": n, "genes": len(d["genes"]), "cells": d["n_obs"]}
                          for n, d in sources.items()]))
    print(f"{len(sources)} source datasets loaded.")

In [ ]:
if sources:
    rows = []
    for r in pairs_c.itertuples():
        has_old = {n for n, d in sources.items() if r.retired   in d["genes"]}
        has_new = {n for n, d in sources.items() if r.successor in d["genes"]}
        rows.append({"retired": r.retired, "successor": r.successor,
                     "with_retired": len(has_old), "with_successor": len(has_new),
                     "with_both": len(has_old & has_new)})
    coexist = pd.DataFrame(rows)
    both = int((coexist.with_both > 0).sum())
    apart = len(coexist) - both

    print(f"Of the {len(coexist):,} group C pairs:\n")
    print(f"  {both:,} have both identifiers in at least one source dataset.")
    print("     Those two genes were counted separately by that study, so its cells may have")
    print("     counts in both columns. The reads are distinct, so adding the columns together")
    print("     does not double-count anything.\n")
    print(f"  {apart:,} never appear together in any dataset.")
    print("     No cell can have counts in both, so a combined column is simple addition.")

## 7. Where did the retired identifiers come from?

Retired identifiers are rarely spread evenly. Seeing which studies carry them is useful context
— and if some studies carry none at all, that is itself informative about how their gene lists
were prepared.

In [ ]:
if sources:
    retired_set = set(retired)
    per_source = (pd.DataFrame([{"dataset": n, "genes": len(d["genes"]),
                                 "retired identifiers": len(d["genes"] & retired_set)}
                                for n, d in sources.items()])
                  .sort_values("retired identifiers", ascending=False))
    display(per_source)
    top = per_source.iloc[0]
    clean = per_source[per_source["retired identifiers"] == 0]
    print(f"{top.dataset} contributes the most ({top['retired identifiers']:,}).")
    if len(clean):
        print(f"{len(clean)} dataset(s) contribute none at all: {', '.join(clean.dataset)}")

## 8. Does each study's declared annotation version match its gene content?

`obs['gene_annotation_version']` records which annotation a study says it used. This is
checkable, because a dataset cannot contain a gene that did not exist yet: we find the earliest
Ensembl release whose gene list accounts for **every** gene in the file.

If that release is later than the one recorded, the file contains genes the recorded release had
not yet defined, and the two disagree.

That disagreement has more than one possible cause — the alignment may have used a later release
than was recorded, or the gene list may have been changed after alignment — so it is a flag to
follow up rather than a conclusion.

A mismatch does not affect any of the groupings above — those come from the atlas's own
identifiers. It does mean the field should not be relied on, and it tells you what to set
`OLD_RELEASE` to in section 1.

Two things to keep in mind when reading the result.

It is a **lower bound**. A file containing a gene first defined in release 96 must have been
built with release 96 or later — but a later release contains those genes too, so this cannot
say which one was actually used.

It depends on testing **consecutive** releases. `RELEASES_TO_TEST` defaults to the full GRCh38
range for that reason; a sparse sample only brackets the answer between two tested values.

In [ ]:
if sources:
    release_genes = {}
    for rel in RELEASES_TO_TEST:
        assembly = 37 if rel < 76 else ENSEMBL_ASSEMBLY
        df = ensembl_query(f"SELECT stable_id FROM {SPECIES}_core_{rel}_{assembly}.gene")
        release_genes[rel] = set(df.stable_id)
    print("Gene counts per Ensembl release: " +
          ", ".join(f"{r}: {len(g):,}" for r, g in release_genes.items()))

In [ ]:
if sources:
    rows = []
    for name, d in sources.items():
        genes = d["genes"]
        earliest = next((r for r in sorted(RELEASES_TO_TEST)
                         if not (genes - release_genes[r])), None)
        rows.append({"dataset": name, "declared": d["declared"], "genes": len(genes),
                     "earliest release containing all genes": earliest})
    versions = pd.DataFrame(rows)
    display(versions)

    def mismatch(row):
        if not row.declared or row["earliest release containing all genes"] is None:
            return False
        try:   return int(str(row.declared).lstrip("v").split(",")[0]) < row["earliest release containing all genes"]
        except ValueError: return False

    bad = versions[versions.apply(mismatch, axis=1)]
    if len(bad):
        print(f"{len(bad)} of {len(versions)} datasets record an Ensembl release whose gene")
        print("catalogue does not contain every gene in the file. In other words, those files")
        print("hold genes that Ensembl had not yet defined at the release they name.\n")
        print("This says the two disagree; it does not say which is wrong. Either the alignment")
        print("used a later release than was recorded, or it used the recorded one and the gene")
        print("list was changed afterwards (lifted, merged, or reindexed during curation).")
        print("Only the producers can say which. Either way the field no longer describes the")
        print("gene axis in the file, so it should not be relied on.")
    else:
        print("Every dataset's recorded release contains all of that file's genes.")

## 9. Summary and output file

The classification is written out here rather than in section 4, so that the file carries the
evidence gathered since: whether each merge was confirmed against the genome, and whether any
source dataset holds both identifiers of a pair. A curator reading only the CSV then sees the
same caveats as a reader of this notebook.

In [ ]:
out = classification.copy()
if "coord_check" in dir():
    out = out.merge(coord_check[["retired", "verdict"]].rename(
        columns={"verdict": "coordinate_check"}), on="retired", how="left")
if sources:
    out = out.merge(coexist[["retired", "with_retired", "with_successor", "with_both"]],
                    on="retired", how="left")
out.to_csv("retired_identifier_classification.csv", index=False)
print(f"Written: retired_identifier_classification.csv "
      f"({len(out):,} rows, {len(out.columns)} columns)\n")

print(f"{ATLAS.name}")
print(f"{adata.n_obs:,} cells x {adata.n_vars:,} genes\n")
print(f"{len(retired):,} gene identifiers are no longer in current GENCODE. Each is an error")
print("in CELLxGENE's validator and must be resolved before submission.\n")
print(f"  A  {len(group_a):>5,}  deleted with no replacement    -> remove the column")
print(f"  B  {len(group_b):>5,}  renamed, successor not present -> rename the column")
print(f"  C  {len(group_c):>5,}  merged, successor already here -> needs a decision\n")
print(f"Splits: {'none' if len(splits) == 0 else str(len(splits)) + ' — review individually'}.")
if len(merged):
    print(f"Merges: {len(merged):,} genes absorb more than one retired identifier "
          f"(up to {merged.max()} at once).")
if sources:
    print(f"\nOf the {len(group_c):,} in group C, {both:,} have both identifiers together in a")
    print(f"source dataset and {apart:,} never do. In both cases the reads behind the two columns")
    print("are distinct, so combining a pair reconstructs the gene as Ensembl now defines it.")
if "coord_check" in dir():
    vc = coord_check.verdict.value_counts()
    contradicted = int(vc.drop(["contained", "not in both releases"], errors="ignore").sum())
    unlocatable  = int(vc.get("not in both releases", 0))
    print(f"\nGenome check on group C: {int(vc.get('contained', 0)):,} confirmed contained, "
          f"{unlocatable:,} not locatable in both releases, {contradicted:,} located but not contained.")
    if contradicted:
        print("The last group is worth reading individually — see section 5 and the")
        print("coordinate_check column in the CSV.")

### A note on CELLxGENE's own tooling

CELLxGENE's migration step remaps identifiers for which a curator has supplied a replacement,
then deletes whatever is left. The remap is a plain index rename with no handling for a target
that already exists, so **it does not resolve group C** — running it would leave two columns
sharing one name. Those cases need deciding before submission, not during it.